# 📘 Databricks Streaming

---

## 🔷 1. What is Streaming? (Start from Basics)

👉 Streaming means **processing data as it arrives (real-time)**

### Example:
- Swiggy orders coming every second 🍔
- Stock prices updating continuously 📈
- Logs generated by apps 🧾

---

### 🆚 Batch vs Streaming

| Type | Data | Example |
|-----|------|--------|
| Batch | Finite | Daily report |
| Streaming | Infinite | Live transactions |

---

## 🔷 2. What is Structured Streaming?

👉 Databricks uses **Structured Streaming (Spark)**

💡 Key Idea:
> Treat streaming data as a **table that keeps growing**

---

### 🧠 Mental Model

```
Table (infinite)
↓
New rows keep coming
↓
Query runs continuously
```

---

### Example:

Batch:
```sql
SELECT COUNT(*) FROM customers;
```

Streaming:
👉 Same query runs continuously as data arrives

---

## 🔷 3. How It Works Internally

👉 Spark does NOT process row-by-row
👉 It uses **micro-batches**

```
Incoming Data → Small Batch → Process → Output → Repeat
```

---

## 🔷 4. Core Components

| Component | Meaning |
|----------|--------|
| Source | Where data comes from |
| Transformation | Business logic |
| Sink | Where data is written |
| Trigger | When processing runs |
| Checkpoint | Saves progress |
| State | Memory for aggregations |

---

## 🔷 5. Reading Streaming Data

---

### 🔹 From Files (Auto Loader)

```python
df = spark.readStream \
  .format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .option("cloudFiles.schemaLocation", "/schema") \
  .load("/input")
```

---

### 🔹 From Kafka

```python
df = spark.readStream \
  .format("kafka") \
  .option("subscribe", "orders") \
  .option("startingOffsets", "earliest") \
  .load()
```

---

### 🔹 From Delta Table

```python
df = spark.readStream.table("sales.analytics.customers")
```

---

### 🔹 SQL Example

```sql
SELECT * FROM STREAM(sales.analytics.customers);
```

---

## 🔷 6. Transformations

👉 Same as batch DataFrame

---

### PySpark

```python
df_filtered = df.filter("amount > 1000")

df_grouped = df.groupBy("country").count()
```

---

### SQL

```sql
SELECT country, COUNT(*)
FROM STREAM(customers)
GROUP BY country;
```

---

## 🔷 7. Writing Streaming Data

---

### PySpark

```python
df.writeStream \
  .format("delta") \
  .option("checkpointLocation", "/chk") \
  .toTable("sales.analytics.output")
```

---

### SQL

```sql
CREATE OR REFRESH STREAMING LIVE TABLE output_table
AS SELECT * FROM STREAM(customers);
```

---

## 🔷 8. Checkpointing (Most Important 🔥)

---

### 🔹 What is it?

👉 Saving progress of streaming job

---

### 🔹 What it stores?

- Offsets (progress)
- State (aggregations)
- Metadata

---

### 🔹 Example

```python
df.writeStream \
  .option("checkpointLocation", "/checkpoint") \
  .start("/output")
```

---

### 🔹 Real Scenario

❌ Without checkpoint:
- Restart → duplicate data

✅ With checkpoint:
- Restart → continues correctly

---

## 🔷 9. Output Modes (VERY IMPORTANT)

---

### 1️⃣ Append Mode

👉 Only new rows

```python
df.writeStream.outputMode("append")
```

---

### 2️⃣ Update Mode

👉 Only changed rows

```python
df.writeStream.outputMode("update")
```

---

### 3️⃣ Complete Mode

👉 Full table every time

```python
df.writeStream.outputMode("complete")
```

---

### 🧠 Easy Understanding

| Mode | Meaning |
|-----|--------|
| Append | Add only new |
| Update | Update changed |
| Complete | Replace all |

---

## 🔷 10. Triggers (Execution Control)

---

### 🔹 Default (Micro-batch)

```python
.trigger(processingTime="10 seconds")
```

👉 Runs every 10 seconds

---

### 🔹 Once

```python
.trigger(once=True)
```

👉 Runs one time only

---

### 🔹 Available Now

```python
.trigger(availableNow=True)
```

👉 Processes all backlog

---

## 🔷 11. Watermark (Late Data Handling)

---

### Problem:
👉 Data arrives late

---

### Solution:

```python
df.withWatermark("event_time", "10 minutes")
```

---

👉 Accept data up to 10 min late

---

## 🔷 12. Window Aggregation

---

### PySpark

```python
from pyspark.sql.functions import window

df.groupBy(window("event_time", "10 minutes")).count()
```

---

### SQL

```sql
SELECT window(event_time, "10 minutes"), COUNT(*)
FROM STREAM(customers)
GROUP BY window(event_time, "10 minutes");
```

---

## 🔷 13. readStream Options (Important 🔥)

---

### Auto Loader Options

```python
.option("cloudFiles.format", "json")
.option("cloudFiles.schemaLocation", "/schema")
.option("cloudFiles.maxFilesPerTrigger", 5)
```

---

### Kafka Options

```python
.option("subscribe", "topic")
.option("startingOffsets", "earliest")
.option("maxOffsetsPerTrigger", 1000)
```

---

### Delta Options

```python
.option("ignoreChanges", "true")
.option("ignoreDeletes", "true")
```

---

## 🔷 14. writeStream Options

---

```python
df.writeStream \
  .format("delta") \
  .option("checkpointLocation", "/chk") \
  .option("mergeSchema", "true") \
  .outputMode("append") \
  .trigger(processingTime="10 seconds")
```

---

## 🔷 15. End-to-End Example (Beginner Friendly)

```python
df = spark.readStream \
  .format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .load("/input")

filtered = df.filter("amount > 1000")

filtered.writeStream \
  .format("delta") \
  .option("checkpointLocation", "/chk") \
  .outputMode("append") \
  .toTable("sales.analytics.high_value_txn")
```

---

## 🔷 16. Common Mistakes 🚨

- ❌ No checkpoint → duplicates
- ❌ Wrong output mode
- ❌ No watermark → memory issues
- ❌ Treating streaming like batch

---

## 🔷 17. Interview Quick Q&A 🎯

👉 What is streaming?
→ Continuous data processing

👉 What is checkpoint?
→ Stores progress

👉 What is watermark?
→ Handles late data

👉 Output modes?
→ Append, Update, Complete

👉 Trigger?
→ Controls execution

---

## 🔷 🧠 Final Mental Model

```
Read → Transform → Write → Checkpoint → Repeat
```

---

## 🔷 🚀 One-Line Summary

> Databricks Streaming = Real-time data processing using Structured Streaming with checkpointing, triggers, and fault tolerance

---


# 📘 Databricks Auto Loader

---

## 🔷 1. What is Auto Loader?

👉 Auto Loader is a feature in Databricks used to **incrementally ingest files from cloud storage**

👉 It is built on **Structured Streaming**

---

### 🧠 Simple Understanding

Without Auto Loader:
- You manually scan folders
- Difficult to track new files

With Auto Loader:
- Automatically detects **new files**
- Processes only **new data**

---

## 🔷 2. Why Auto Loader?

### Problems it solves:

- ❌ Re-reading same files
- ❌ Missing new files
- ❌ Poor scalability
- ❌ Manual file tracking

---

### Benefits:

- ✅ Incremental ingestion
- ✅ Scalable to millions of files
- ✅ Fault-tolerant
- ✅ Schema evolution support

---

## 🔷 3. How Auto Loader Works

👉 Uses **file notification OR directory listing**

---

### 🔹 1. Directory Listing Mode

- Scans directory periodically
- Simpler but slower

---

### 🔹 2. File Notification Mode (Recommended 🚀)

- Uses cloud notifications (AWS SQS, Azure Queue)
- Faster and scalable

---

## 🔷 4. Basic Syntax

```python
df = spark.readStream \
  .format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .load("/input")
```

---

## 🔷 5. Required Options

### 🔹 cloudFiles.format

👉 Defines file format

```python
.option("cloudFiles.format", "json")
```

Supported:

- json
- csv
- parquet
- avro

---

### 🔹 cloudFiles.schemaLocation (IMPORTANT 🔥)

👉 Stores schema metadata

```python
.option("cloudFiles.schemaLocation", "/schema")
```

---

## 🔷 6. Schema Handling Options

### 🔹 Infer Schema

```python
.option("cloudFiles.inferColumnTypes", "true")
```

---

### 🔹 Provide Schema Manually

```python
schema = "id INT, name STRING"

df = spark.readStream \
  .format("cloudFiles") \
  .schema(schema) \
  .load("/input")
```

---

### 🔹 Schema Evolution

```python
.option("cloudFiles.schemaEvolutionMode", "addNewColumns")
```

Modes:

- addNewColumns
- rescue
- failOnNewColumns

---

## 🔷 7. File Processing Options

### 🔹 maxFilesPerTrigger

👉 Controls batch size

```python
.option("cloudFiles.maxFilesPerTrigger", 10)
```

---

### 🔹 includeExistingFiles

👉 Process old files

```python
.option("cloudFiles.includeExistingFiles", "true")
```

---

### 🔹 allowOverwrites

```python
.option("cloudFiles.allowOverwrites", "true")
```

---

## 🔷 8. Notification Mode Options

### 🔹 useNotifications

```python
.option("cloudFiles.useNotifications", "true")
```

---

### 🔹 queueName (Azure / AWS)

```python
.option("cloudFiles.queueName", "my-queue")
```

---

## 🔷 9. Data Quality Options

### 🔹 Bad Records Path

```python
.option("badRecordsPath", "/bad_records")
```

---

### 🔹 Rescue Data (Corrupt Data)

```python
.option("rescuedDataColumn", "_rescued_data")
```

---

## 🔷 10. Partition Handling

### 🔹 Partition Columns

```python
.option("cloudFiles.partitionColumns", "date,region")
```

---

## 🔷 11. Write with Auto Loader

```python
df.writeStream \
  .format("delta") \
  .option("checkpointLocation", "/chk") \
  .toTable("sales.analytics.table")
```

---

## 🔷 12. Full Example (End-to-End)

```python
df = spark.readStream \
  .format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .option("cloudFiles.schemaLocation", "/schema") \
  .option("cloudFiles.inferColumnTypes", "true") \
  .option("cloudFiles.maxFilesPerTrigger", 5) \
  .load("/input")

df.writeStream \
  .format("delta") \
  .option("checkpointLocation", "/chk") \
  .option("mergeSchema", "true") \
  .toTable("sales.analytics.autoloader_data")
```

---

## 🔷 13. SQL Example

```sql
CREATE OR REFRESH STREAMING LIVE TABLE auto_loader_table
AS
SELECT *
FROM cloud_files("/input", "json");
```

---

## 🔷 14. Best Practices 🚀

- Always use schemaLocation
- Use notification mode for large data
- Limit batch size using maxFilesPerTrigger
- Use checkpointing
- Handle corrupt data properly

---

## 🔷 15. Common Mistakes 🚨

- ❌ Missing schemaLocation
- ❌ Not using checkpoint
- ❌ Large batch size causing failure
- ❌ Ignoring schema evolution

---

## 🔷 16. Interview Questions 🎯

👉 What is Auto Loader?
→ Incremental file ingestion system

👉 Directory vs Notification mode?
Directory → scans files
Notification → event-based

👉 Why schemaLocation?
→ Stores schema for consistency

👉 What is schema evolution?
→ Handles new columns automatically

---

## 🔷 🧠 Final Mental Model

```
New Files → Auto Loader → Process → Delta Table → Checkpoint
```

---

## 🔷 🚀 One-Line Summary

Auto Loader = Scalable, incremental file ingestion system built on Structured Streaming


# 📊 Structured Streaming: Reliable Progress Tracking (Apache Spark)

## 🧠 Overview
Apache Spark Structured Streaming ensures **fault-tolerant and reliable stream processing** by tracking progress using:
- Offset management
- Checkpointing
- Micro-batch execution
- Write-ahead logs
- Idempotent / transactional sinks

It enables **exactly-once processing (in most real-world scenarios)**.

---

# 📍 1. Offset Tracking (Input Progress)

Structured Streaming tracks **how much data has been consumed** using *offsets*.

### Examples:
- Kafka → Offset = message position
- File source → Tracks processed files

### Flow:
1. Read data from offset `X`
2. Process batch
3. Store offset range `X → Y` in checkpoint

✔️ Guarantees:
- No data loss
- No skipping

👉 Progress = *All data up to offset Y processed*

---

# 📍 2. Checkpointing (Core Mechanism)

Checkpoint directory stores:
- Offsets
- State (aggregations, joins)
- Metadata (batch IDs)

### Storage:
- HDFS / S3 / DBFS

### Why important:
- Enables restart from failure point
- Avoids recomputation

---

# 📍 3. Micro-Batch Execution

Processing happens in small batches:

- Batch 1 -> Batch 2 -> Batch 3


Each batch:
- Has unique ID
- Has fixed input range

👉 Progress = last successful batch ID

---

# 📍 4. Write-Ahead Log (WAL)

Before execution:
- Batch details are logged

### Benefit:
- Safe recovery
- Replay capability

---

# 📍 5. Exactly-Once Guarantee

Achieved using:

### ✅ Deterministic computation
Same input → same output

### ✅ Sink guarantees
- Idempotent (safe retry)
- Transactional (ACID)

### Examples:
- Delta Lake
- Kafka

---

# 📍 6. Output Commit Protocol

Ensures:
- Data is written only once per batch

Even with:
- Failures
- Retries

Uses:
- Batch IDs
- Commit logs

---

# 🔁 Failure Scenario

### Example:
- Batch 10 fails mid-way

### Recovery:
- Restart from Batch 9
- Reprocess Batch 10

✔️ No duplication
✔️ No loss

---

# 🧩 Architecture Flow
- Source → Offsets → Micro-batch → Processing → Sink
- ↓
- Checkpoint

---

# ⚡ Key Insight

Structured Streaming tracks:
- ❌ Not individual records
- ✅ Offset ranges per batch

👉 This makes it scalable + reliable

---

# ✅ Summary Table

| Component        | Role                          |
|----------------|-------------------------------|
| Offsets         | Track input progress          |
| Checkpoint      | Recovery + state              |
| Batch IDs       | Execution tracking            |
| WAL             | Replay safety                 |
| Commit Protocol | Exactly-once writes           |

---

# 🚀 Final Takeaway

Structured Streaming ensures reliability through:
- Offset tracking
- Checkpointing
- Deterministic execution
- Fault-tolerant sinks

👉 Result:
**Scalable + Fault-tolerant + Near Exactly-once Processing**

---

# 📊 Structured Streaming Notes: State Management & Join Types

## 🧠 Overview

In Apache Spark Structured Streaming, state management is critical for
performance and scalability.

------------------------------------------------------------------------

# ⚡ Stateless vs Stateful Processing

## Stateless Processing

-   No dependency on past data
-   Examples: SELECT, WHERE, map
-   No state storage
-   Highly scalable

## Stateful Processing

-   Depends on historical data
-   Examples: Aggregations, joins, deduplication
-   Uses state store
-   Can grow unbounded

------------------------------------------------------------------------

# 🚨 State Explosion Problem

State grows due to: - High cardinality keys - No watermark - Outer
joins - Long windows

------------------------------------------------------------------------

# 🎯 Techniques to Limit State

## Watermarking

WITH WATERMARK (event_time, '10 minutes')

## Windowed Aggregation

GROUP BY user_id, window(event_time, '10 minutes')

## Reduce Key Cardinality

Use grouped dimensions instead of unique keys

## Deduplication

DROP DUPLICATES WITH WATERMARK

## Prefer Stream-Static Join

Reduces state significantly

------------------------------------------------------------------------

# 🔗 Join Types & State Impact

## Stream-Static Join (Best)

State: LOW

## Stream-Stream Inner Join

State: MEDIUM

## Left Outer Join

State: HIGH

## Right Outer Join

State: HIGH

## Full Outer Join (Worst)

State: VERY HIGH

## Cross Join

Not supported

------------------------------------------------------------------------

# 📉 Watermark Importance

Without watermark → infinite state\
With watermark → bounded state

------------------------------------------------------------------------

# 📊 Summary Table

  Join Type             State Size
  --------------------- ------------
  Stream-Static         LOW
  Stream-Stream Inner   MEDIUM
  Left Outer            HIGH
  Right Outer           HIGH
  Full Outer            VERY HIGH

------------------------------------------------------------------------

# 🚀 Final Takeaway

Use stateless operations, watermarking, and stream-static joins to
control state.
